In [ ]:
import csv

# Function to read the CSV file and convert it to the desired format
def read_csv_to_dict(file_path):
    program_ratings = {}

    with open(file_path, mode='r', newline='') as file:
        reader = csv.reader(file)
        # Skip the header
        header = next(reader)

        for row in reader:
            program = row[0]
            ratings = [float(x) for x in row[1:]]  # Convert the ratings to floats
            program_ratings[program] = ratings

    return program_ratings

# Path to the CSV file
file_path = 'program_ratings_modified.csv'

# Get the data in the required format
program_ratings_dict = read_csv_to_dict(file_path)

# Print the result (you can also return or process it further)
for program, ratings in program_ratings_dict.items():
    print(f"'{program}': {ratings},")


'news': [0.7, 0.7, 0.8, 0.3, 0.5, 0.4, 0.3, 0.2, 0.1, 0.2, 0.3, 0.7, 0.8, 0.7, 0.3, 0.2, 0.1, 0.2],
'live_soccer': [0.0, 0.0, 0.0, 0.2, 0.1, 0.3, 0.2, 0.1, 0.4, 0.3, 0.4, 0.5, 0.4, 0.6, 0.9, 1.0, 0.9, 0.3],
'movie_a': [0.1, 0.1, 0.2, 0.4, 0.3, 0.2, 0.1, 0.2, 0.3, 0.4, 0.5, 0.4, 0.3, 0.4, 0.8, 0.9, 0.8, 0.4],
'movie_b': [0.2, 0.1, 0.1, 0.3, 0.2, 0.1, 0.2, 0.3, 0.4, 0.5, 0.4, 0.3, 0.4, 0.5, 0.7, 0.8, 0.8, 0.7],
'reality_show': [0.3, 0.4, 0.3, 0.4, 0.6, 0.7, 0.6, 0.6, 0.7, 0.6, 0.3, 0.2, 0.1, 0.2, 0.3, 0.2, 0.2, 0.3],
'tv_series_a': [0.2, 0.3, 0.2, 0.1, 0.1, 0.2, 0.2, 0.4, 0.4, 0.3, 0.3, 0.3, 0.7, 0.8, 0.8, 0.8, 0.4, 0.3],
'tv_series_b': [0.1, 0.2, 0.3, 0.3, 0.2, 0.3, 0.3, 0.1, 0.4, 0.3, 0.4, 0.3, 0.7, 0.7, 0.8, 0.8, 0.4, 0.3],
'music_program': [0.3, 0.3, 0.3, 0.2, 0.2, 0.6, 0.6, 0.7, 0.6, 0.6, 0.3, 0.3, 0.2, 0.3, 0.2, 0.3, 0.5, 0.3],
'documentary': [0.3, 0.3, 0.6, 0.6, 0.6, 0.2, 0.3, 0.4, 0.4, 0.3, 0.2, 0.2, 0.2, 0.1, 0.1, 0.3, 0.3, 0.2],
'Boxing': [0.4, 0.3, 0.3, 0.2, 0.2, 0.1, 0.1, 0.1

In [7]:
import random

##################################### DEFINING PARAMETERS AND DATASET ################################################################
# Sample rating programs dataset for each time slot.
ratings = program_ratings_dict

GEN = 100
POP = 50
CO_R = 0.8
MUT_R = 0.2
EL_S = 2

all_programs = list(ratings.keys()) # all programs
all_time_slots = list(range(6, 24)) # time slots

######################################### DEFINING FUNCTIONS ########################################################################
# defining fitness function
def fitness_function(schedule):
    total_rating = 0
    for time_slot, program in enumerate(schedule):
        total_rating += ratings[program][time_slot]
    return total_rating

# initializing the population
def initialize_pop(programs, time_slots):
    if not programs:
        return [[]]

    all_schedules = []
    for i in range(len(programs)):
        for schedule in initialize_pop(programs[:i] + programs[i + 1:], time_slots):
            all_schedules.append([programs[i]] + schedule)

    return all_schedules

# selection
def finding_best_schedule(all_schedules):
    best_schedule = []
    max_ratings = 0

    for schedule in all_schedules:
        total_ratings = fitness_function(schedule)
        if total_ratings > max_ratings:
            max_ratings = total_ratings
            best_schedule = schedule

    return best_schedule

# calling the pop func.
all_possible_schedules = initialize_pop(all_programs, all_time_slots)

# callin the schedule func.
best_schedule = finding_best_schedule(all_possible_schedules)


############################################# GENETIC ALGORITHM #############################################################################

# Crossover
def crossover(schedule1, schedule2):
    crossover_point = random.randint(1, len(schedule1) - 2)
    child1 = schedule1[:crossover_point] + schedule2[crossover_point:]
    child2 = schedule2[:crossover_point] + schedule1[crossover_point:]
    return child1, child2

# mutating
def mutate(schedule):
    mutation_point = random.randint(0, len(schedule) - 1)
    new_program = random.choice(all_programs)
    schedule[mutation_point] = new_program
    return schedule

# calling the fitness func.
def evaluate_fitness(schedule):
    return fitness_function(schedule)

# genetic algorithms with parameters



def genetic_algorithm(initial_schedule, generations=GEN, population_size=POP, crossover_rate=CO_R, mutation_rate=MUT_R, elitism_size=EL_S):

    population = [initial_schedule]

    for _ in range(population_size - 1):
        random_schedule = initial_schedule.copy()
        random.shuffle(random_schedule)
        population.append(random_schedule)

    for generation in range(generations):
        new_population = []

        # Elitsm
        population.sort(key=lambda schedule: fitness_function(schedule), reverse=True)
        new_population.extend(population[:elitism_size])

        while len(new_population) < population_size:
            parent1, parent2 = random.choices(population, k=2)
            if random.random() < crossover_rate:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = parent1.copy(), parent2.copy()

            if random.random() < mutation_rate:
                child1 = mutate(child1)
            if random.random() < mutation_rate:
                child2 = mutate(child2)

            new_population.extend([child1, child2])

        population = new_population

    return population[0]

##################################################### RESULTS ###################################################################################


# RESULTS - THREE EXPERIMENTS

experiments = [
    ("Experiment 1", 0.80, 0.20),
    ("Experiment 2", 0.70, 0.01),
    ("Experiment 3", 0.95, 0.05)
]

initial_best_schedule = finding_best_schedule(
    all_possible_schedules
)

rem_t_slots = len(all_time_slots) - len(initial_best_schedule)

results = []
experiment_schedules = {}

for name, co_rate, mut_rate in experiments:
    random.seed(42)

    genetic_schedule = genetic_algorithm(
        initial_best_schedule,
        generations=GEN,
        population_size=POP,
        crossover_rate=co_rate,
        mutation_rate=mut_rate,
        elitism_size=EL_S
    )

    final_schedule = (
        initial_best_schedule +
        genetic_schedule[:rem_t_slots]
    )

    total_rating = fitness_function(final_schedule)
    experiment_schedules[name] = final_schedule.copy()

    results.append((name, co_rate, mut_rate, total_rating))

    print(f"\n{name}")
    print(f"Crossover Rate: {co_rate}")
    print(f"Mutation Rate: {mut_rate}")

    for time_slot, program in enumerate(final_schedule):
        print(
            f"Time Slot {all_time_slots[time_slot]:02d}:00"
            f" - Program {program}"
        )

    print(f"Total Ratings: {total_rating:.2f}")
    print("-" * 50)

print("\nEXPERIMENT COMPARISON")
for name, co_rate, mut_rate, rating in results:
    print(
        f"{name}: CO_R={co_rate}, "
        f"MUT_R={mut_rate}, Total={rating:.2f}"
    )



Experiment 1
Crossover Rate: 0.8
Mutation Rate: 0.2
Time Slot 06:00 - Program Boxing
Time Slot 07:00 - Program tv_series_a
Time Slot 08:00 - Program news
Time Slot 09:00 - Program movie_a
Time Slot 10:00 - Program documentary
Time Slot 11:00 - Program reality_show
Time Slot 12:00 - Program tv_series_b
Time Slot 13:00 - Program music_program
Time Slot 14:00 - Program live_soccer
Time Slot 15:00 - Program movie_b
Time Slot 16:00 - Program news
Time Slot 17:00 - Program news
Time Slot 18:00 - Program news
Time Slot 19:00 - Program documentary
Time Slot 20:00 - Program news
Time Slot 21:00 - Program music_program
Time Slot 22:00 - Program music_program
Time Slot 23:00 - Program reality_show
Total Ratings: 8.40
--------------------------------------------------

Experiment 2
Crossover Rate: 0.7
Mutation Rate: 0.01
Time Slot 06:00 - Program Boxing
Time Slot 07:00 - Program tv_series_a
Time Slot 08:00 - Program news
Time Slot 09:00 - Program movie_a
Time Slot 10:00 - Program documentary
Time

In [8]:

import pandas as pd

# Display schedules for all three experiments
for name, schedule in experiment_schedules.items():

    print(f"\n{name} - TV Programming Schedule")

    schedule_data = []

    for i, program in enumerate(schedule):
        start_hour = all_time_slots[i]
        end_hour = (start_hour + 1) % 24

        schedule_data.append({
            "Time": f"{start_hour:02d}:00 - {end_hour:02d}:00",
            "TV Program": program.replace("_", " ").title()
        })

    schedule_df = pd.DataFrame(schedule_data)

    display(schedule_df)



Experiment 1 - TV Programming Schedule


,Time,TV Program
0,06:00 - 07:00,Boxing
1,07:00 - 08:00,Tv Series A
2,08:00 - 09:00,News
3,09:00 - 10:00,Movie A
4,10:00 - 11:00,Documentary
5,11:00 - 12:00,Reality Show
6,12:00 - 13:00,Tv Series B
7,13:00 - 14:00,Music Program
8,14:00 - 15:00,Live Soccer
9,15:00 - 16:00,Movie B



Experiment 2 - TV Programming Schedule


,Time,TV Program
0,06:00 - 07:00,Boxing
1,07:00 - 08:00,Tv Series A
2,08:00 - 09:00,News
3,09:00 - 10:00,Movie A
4,10:00 - 11:00,Documentary
5,11:00 - 12:00,Reality Show
6,12:00 - 13:00,Tv Series B
7,13:00 - 14:00,Music Program
8,14:00 - 15:00,Live Soccer
9,15:00 - 16:00,Movie B



Experiment 3 - TV Programming Schedule


,Time,TV Program
0,06:00 - 07:00,Boxing
1,07:00 - 08:00,Tv Series A
2,08:00 - 09:00,News
3,09:00 - 10:00,Movie A
4,10:00 - 11:00,Documentary
5,11:00 - 12:00,Reality Show
6,12:00 - 13:00,Tv Series B
7,13:00 - 14:00,Music Program
8,14:00 - 15:00,Live Soccer
9,15:00 - 16:00,Movie B
